# 針対平板（軸対称）のストリーマー

上から円錐形の針電極を吊るし、下端（z = 0）を接地平板とした軸対称の計算です。

```
 z = L ┌──────┬──┬──────┐   L = ギャップ + 針の長さ
       │      │  │      │   針：円柱（shank_radius）→ 円錐（cone_length）→ 丸い先端（tip_radius）
       │       \/       │
 z = d │       ・ ← 先端  │   d = ギャップ
       │                │
 z = 0 └────────────────┘   接地平板
       r = 0（中心軸）
```

* 元の設定：`programs/standard_2d/streamer_cyl_electrode.cfg`（空気、イオンも動く）
* 反応セット（`CHEMISTRY`）
  * `"simple"`：電離と付着だけ（`td_air_siglo_swarm.txt`）
  * `"detailed"`：`air_chemistry_v3`（電子の脱離・イオンの変換・再結合を含む 39 反応）に、
    発光する N2(C3) の生成・消光・放射（4 反応）を加えた `transport_data/air_chemistry_v3_light.txt`。
    二次ストリーマーや発光（ストリーク画像）を見るときはこちら
* 物理的な条件（ギャップ・電圧など）から設定への変換は `afivo_helpers.needle_plane_settings()` が行う
* 公式の例のうち `fixes%source_factor = original_flux` は今のコードで使えないので `flux` にしている

**計算時間の目安**（このPC、4 スレッド、ギャップ 10 mm・先端半径 200 μm・±20 kV）
* 正極性の針：1 ns あたり約 1 分（`"detailed"` では約 2.5 分）
* 橋絡のあと：1 ns あたり十数分以上（平板の表面に強い電界の層ができ、時間刻みが小さくなるため）
* 負極性の針：1 ns あたり約 9 分（電極表面の電界が強くなり時間刻みが小さくなるため）
* 先端を鋭くする（`tip_radius` を小さくする）と格子が細かくなり、さらに重くなる

## 1. 条件

In [ ]:
# ===== ここを変える =====
NAME = "np_pos_20kV_chem"     # 計算の名前（結果は programs/standard_2d/output/<NAME>_* ）
GAP = 10e-3                   # ギャップ（針の先端〜平板）(m)
VOLTAGE = 20e3                # 針の電圧 (V)。正 → 正極性の針、負 → 負極性の針
TIP_RADIUS = 200e-6           # 先端の曲率半径 (m)
SHANK_RADIUS = 0.5e-3         # 針の太さ（円柱部分の半径）(m)
CONE_LENGTH = 2e-3            # 円錐部分の長さ (m)
NEEDLE_LENGTH = 4e-3          # 計算領域に入っている針の長さ (m)
PHOTOI = True                 # 光電離
BACKGROUND = 1e13             # 初期の電子・イオン密度 (m^-3)
CHEMISTRY = "detailed"        # "simple" または "detailed"（発光・二次ストリーマー用）
ION_MOBILITY = 2.2e-4         # イオンの移動度 (m^2/(V s))、"detailed" のとき
END_TIME = 15e-9              # 終了時刻 (s)。STOP_AT_BRIDGE なら、その前に止まる
STOP_AT_BRIDGE = True         # 頭が平板に近づいたら止める（橋絡）。
                              # 二次ストリーマーを見るには False にして END_TIME を 50e-9 などに（数時間以上かかる）
BRIDGE_MARGIN = 0.2e-3        # 平板からこの距離まで来たら止める (m)
OUTPUT_DT = 0.2e-9            # 出力間隔 (s)
THREADS = 4
# ========================

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja  # 日本語フォント
import numpy as np
import pandas as pd

from matplotlib.colors import LogNorm
from afivo_helpers import (REPO, run_streamer, load_log, silo_files, load_silo,
                           front_velocity, lineout_settings, load_lineout,
                           needle_plane_settings, streak_image, N2_C3_DECAY_RATE)

overrides, geom = needle_plane_settings(
    gap=GAP, voltage=VOLTAGE, tip_radius=TIP_RADIUS, shank_radius=SHANK_RADIUS,
    cone_length=CONE_LENGTH, needle_length=NEEDLE_LENGTH, photoi=PHOTOI,
    background_density=BACKGROUND, chemistry=CHEMISTRY, ion_mobility=ION_MOBILITY)
overrides.update({"end_time": END_TIME, "output%dt": OUTPUT_DT,
                  **lineout_settings(2, ("e", "electric_fld", "phi"))})
if STOP_AT_BRIDGE:
    # 最大電界の位置（正極性では頭）が、最初の位置からこの距離だけ動いたら止まる
    overrides.update({"use_end_streamer_length": True,
                      "end_streamer_length": GAP - BRIDGE_MARGIN})

print(f"極性: {'正' if VOLTAGE > 0 else '負'}極性の針, 電圧 {VOLTAGE/1e3:+.1f} kV, 反応セット: {CHEMISTRY}")
print(f"計算領域 {geom['L']*1e3:.1f} mm 四方, ギャップ {GAP*1e3:.1f} mm, 平均電界 {geom['mean_field']/1e6:.2f} MV/m")
print(f"円錐の半頂角 {geom['cone_half_angle_deg']:.1f} 度, 電極付近の格子 {overrides['refine_electrode_dx']*1e6:.0f} μm")
pd.Series(overrides, name="設定").to_frame()

## 2. 計算

同じ名前で最後まで計算済みなら、読み込むだけです（`reuse=True`）。条件を変えたら `NAME` も変えてください。

In [ ]:
base = run_streamer("standard_2d", "streamer_cyl_electrode.cfg", NAME, overrides,
                    threads=THREADS, show_every=10, reuse=True)
log = load_log(base)
T_LAST = log["time"].iloc[-1]
bridged = "reached its desired length" in open(f"{base}_stdout.txt", errors="ignore").read()
print(f"最後の出力: t = {T_LAST*1e9:.2f} ns", "（橋絡の手前で停止）" if bridged else "")

## 3. 初期の電界（電荷がない状態の電界）

t = 0 の出力から、中心軸上の電界と電位を見ます。先端で電界がどれだけ強まっているかの目安になります。

In [ ]:
df0, t0 = load_lineout(base, 0)
gas = df0["y"] < GAP                      # 先端より下（気体の部分）
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), constrained_layout=True)
ax[0].plot(df0["y"][gas] * 1e3, df0["electric_fld"][gas] / 1e6)
ax[0].axhline(geom["mean_field"] / 1e6, color="gray", ls="--", lw=1, label="平均電界 |V|/d")
ax[0].axhline(3.0, color="C3", ls=":", lw=1, label="$E_k$ ≈ 3 MV/m")
ax[0].set_ylabel("電界 (MV/m)"); ax[0].legend()
ax[1].plot(df0["y"] * 1e3, df0["phi"] / 1e3)
ax[1].set_ylabel("電位 (kV)")
for a in ax:
    a.set_xlabel("z (mm)"); a.grid(alpha=0.3)
fig.suptitle("t = 0, 中心軸上")
E_tip0 = df0["electric_fld"][gas].iloc[-1]
print(f"先端の電界 {E_tip0/1e6:.1f} MV/m（平均電界の {E_tip0/geom['mean_field']:.0f} 倍）")
z_k = df0["y"][gas][df0["electric_fld"][gas] > 3e6]
if len(z_k):
    print(f"E > 3 MV/m の領域: 先端から {(GAP - z_k.min())*1e3:.2f} mm")

## 4. 時間変化

In [ ]:
t_ns = log["time"] * 1e9
head = log["ne_zmin"]                     # 電子密度がしきい値を超える最小の z ＝ 頭の位置
# 頭の電界：最大電界 max(E) のうち、その位置 (y) が頭の近く（0.5 mm 以内）にあるもの。
# max(Etip) は計算領域の境界を避けるため、頭が平板に近づくと別の場所を指すので使わない。
at_head = (log["y"] - head).abs() < 0.5e-3
E_head = log["max(E)"].where(at_head)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
ax[0].plot(t_ns[1:], head[1:] * 1e3)
ax[0].axhline(GAP * 1e3, color="gray", ls="--", lw=1)
ax[0].set_ylabel("頭の位置 z (mm)")
ax[1].plot(t_ns, E_head / 1e6, label="頭の電界")
ax[1].plot(t_ns, log["max(E)"] / 1e6, ls=":", label="全体の最大（t=0 は電極表面）")
ax[1].set_ylabel("電界 (MV/m)"); ax[1].legend()
v = np.gradient(head, log["time"])
ax[2].plot(t_ns[2:], -v[2:] / 1e3)
ax[2].set_ylabel("速度 (km/s)")
for a in ax:
    a.set_xlabel("t (ns)"); a.grid(alpha=0.3)

v_mean = (GAP - head.iloc[-1]) / T_LAST
print(f"{T_LAST*1e9:.2f} ns での頭の位置: z = {head.iloc[-1]*1e3:.2f} mm（ギャップの {100*(GAP-head.iloc[-1])/GAP:.0f}% まで進展）")
print(f"平均速度（ギャップ横断）: {v_mean/1e3:.0f} km/s,  頭の電界（後半平均）: {E_head[log['time'] > 0.5*T_LAST].mean()/1e6:.1f} MV/m")

## 5. 分布（電子密度と電界）

In [ ]:
files = silo_files(base)
pick = [files[len(files) * k // 4] for k in (1, 2, 3)] + [files[-1]]
L = geom["L"]
rmax = min(L, 4e-3)
fig, axes = plt.subplots(2, len(pick), figsize=(3.2 * len(pick), 9), constrained_layout=True)
for j, f in enumerate(pick):
    lsf, r, z, t = load_silo(f, "lsf", r_max=[rmax, L])
    for i, (var, cmap, vmax, lab) in enumerate([("e", "inferno", 3e20, "電子密度 (m$^{-3}$)"),
                                                ("electric_fld", "viridis", 2e7, "電界 (V/m)")]):
        data = load_silo(f, var, r_max=[rmax, L])[0]
        ax = axes[i, j]
        mirror = lambda a: np.concatenate([a[::-1], a]).T
        ext = [-r[-1] * 1e3, r[-1] * 1e3, z[0] * 1e3, z[-1] * 1e3]
        im = ax.imshow(mirror(data), origin="lower", extent=ext, cmap=cmap,
                       vmin=0, vmax=vmax, aspect="equal")
        rr = np.linspace(ext[0], ext[1], 2 * len(r)); zz = np.linspace(ext[2], ext[3], len(z))
        ax.contour(rr, zz, mirror(lsf), [0], colors="w", linewidths=1)
        if i == 0:
            ax.set_title(f"t = {t*1e9:.1f} ns")
        ax.set_xlabel("r (mm)")
    axes[0, j].set_ylabel("z (mm)"); axes[1, j].set_ylabel("z (mm)")
fig.colorbar(axes[0, -1].images[0], ax=axes[0, :], shrink=0.8).set_label("電子密度 (m$^{-3}$)")
fig.colorbar(axes[1, -1].images[0], ax=axes[1, :], shrink=0.8).set_label("電界 (V/m)")

## 6. ストリーク画像（発光）

N2(C3) の放射（第 2 正帯, 波長 300〜400 nm）を発光とみなし、発光密度を
$\varepsilon = A\,n_{\mathrm{N_2(C^3)}}$（$A = 2.38\times10^7$ s$^{-1}$、寿命 42 ns）とする。

* **スリット（`mode="slit"`）**：中心軸を通る視線で積分した明るさ $I(z) = 2\int_0^R \varepsilon\,dr$。
  スリットを放電の軸に合わせたストリークカメラに相当する
* **全体（`mode="total"`）**：半径方向にすべて足した $I(z) = \int_0^R \varepsilon\, 2\pi r\,dr$

横軸が時間、縦軸が位置。明るさは何桁も変わるので対数で表示する。
時間の分解能は出力間隔（`OUTPUT_DT`）で決まる。
`CHEMISTRY = "simple"` では N2(C3) を計算しないので、この節は省略される。

In [ ]:
if CHEMISTRY != "detailed":
    print('CHEMISTRY = "detailed" のときだけ描きます')
else:
    t_s, z_s, I_s = streak_image(base, r_max=min(geom["L"], 4e-3), z_max=GAP, mode="slit")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True,
                           gridspec_kw={"width_ratios": [3, 1]})
    vmax = I_s.max()
    dt = t_s[1] - t_s[0]
    cmap = plt.get_cmap("inferno").copy(); cmap.set_bad("black")   # 発光ゼロ（対数で表せない）を黒に
    im = ax[0].imshow(np.ma.masked_less_equal(I_s.T, 0), origin="lower", aspect="auto", cmap=cmap,
                      extent=[(t_s[0] - dt/2)*1e9, (t_s[-1] + dt/2)*1e9, z_s[0]*1e3, z_s[-1]*1e3],
                      norm=LogNorm(vmin=vmax * 1e-3, vmax=vmax))
    ax[0].plot(log["time"][1:] * 1e9, log["ne_zmin"][1:] * 1e3, "c--", lw=1, label="頭の位置（ne_zmin）")
    ax[0].set_xlabel("t (ns)"); ax[0].set_ylabel("z (mm)")
    ax[0].set_title("ストリーク画像（中心軸を通る視線の発光, 対数）")
    ax[0].legend(loc="lower left")
    fig.colorbar(im, ax=ax[0]).set_label("明るさ (photons m$^{-2}$ s$^{-1}$)")

    # 時間で積分した発光の軸方向分布（シャッターを開けっぱなしにした写真に相当）
    ax[1].semilogx(I_s.sum(axis=0) * dt, z_s * 1e3)
    ax[1].set_xlabel("時間積分した明るさ (photons m$^{-2}$)")
    ax[1].set_ylim(z_s[0] * 1e3, z_s[-1] * 1e3); ax[1].grid(alpha=0.3)
    ax[1].set_title("時間積分")

## 7. 次の使い方

* **電圧や先端半径を変える**：1 節の値と `NAME` を変えて上から実行する
* **複数の条件を比べる**：`needle_plane_settings()` と `run_streamer(..., reuse=True)` をループで回す
  （例：`for V in [15e3, 20e3, 25e3]: ...`）
* **負極性**：`VOLTAGE` を負にする。計算時間は正極性の約 9 倍
* **二次ストリーマー**：`CHEMISTRY = "detailed"`、`STOP_AT_BRIDGE = False`、`END_TIME = 50e-9` などにする。
  橋絡のあとは 1 ns あたり十数分以上かかるので、数時間〜半日かかる。ストリーク画像で、針の近くから
  遅れて伸びる発光として見えるはず
* **格子の確認**：結論を出す前に `refine_adx`（標準 1）を 0.5 にして結果が変わらないか確かめる
  （空気・2 MV/m の棒なしの計算では、標準の格子で速度が約 3% 低く出た。`docs/results/grid_2d`）